<h1><span style="color:red">Spatial ESDA and OLS Regression</span></h1>

Exploratory Spatial Data Analysis: spatial weights, spatial lag, join counts, global and local Moran's I (LISA), and OLS spatial regression.

## 1. Retrieve survey parameters from the URL

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


In [ ]:
# common imports
from __future__ import print_function
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

from shapely.geometry import Point
import libpysal
from libpysal.weights import Queen, KNN
import esda
from esda.moran import Moran, Moran_Local
from splot.esda import plot_moran
from splot.libpysal import plot_spatial_weights

pd.set_option('display.max_colwidth', 0)

def printmd(string):
    display(Markdown(string))

# local imports
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint


## 2. Read the survey file and extract numeric variables

In [ ]:

# read the csv file
df = panellibs.extract_data(absolutePath + csv_file)# print(absolutePath + csv_file)

# create a list of variable names
variables_df = pd.DataFrame({'varname':df.columns})
printmd("<b><span style='color:red'>All variables in the survey file:</span></b>")

col = 0
for var in variables_df.varname.values:
    print(str(col) +":  "+ var)
    col = col+1


# create a dictionary of #number variables with abbreviated and full variable names 
var_list = {n[:n.index('#')]:n for n in variables_df.varname.values if '#number' in n}
printmd("<b><span style='color:red'>Numeric variables:</span></b>")

col = 0
for var in var_list:
    print(str(col) +":  "+ var)
    col = col+1

#create a dataframe of only #number variables
nums_df = df[[n for n in variables_df.varname.values if '#number' in n]]


## 3. Detect Geometry and Create GeoDataFrame

Supports WKT geometry columns and point data via Latitude/Longitude columns.

In [ ]:
df.columns = df.columns.str.strip()

geometry_col = next((c for c in df.columns if 'geometry' in c.lower()), None)
lat_col      = next((c for c in df.columns if 'latitude'  in c.lower()), None)
lon_col      = next((c for c in df.columns if 'longitude' in c.lower()), None)

if geometry_col:
    gdf = gpd.GeoDataFrame(
        df, geometry=gpd.GeoSeries.from_wkt(df[geometry_col]), crs='EPSG:4326')
    print(f'Geometry: WKT column "{geometry_col}"')
elif lat_col and lon_col:
    _df = df.dropna(subset=[lat_col, lon_col]).copy()
    _df['geometry'] = _df.apply(
        lambda r: Point(r[lon_col], r[lat_col]), axis=1)
    gdf = gpd.GeoDataFrame(_df, geometry='geometry', crs='EPSG:4326')
    print(f'Geometry: lat={lat_col!r}, lon={lon_col!r}')
else:
    from IPython.display import HTML as _H
    display(_H('<div style="background:#dc3545;color:white;padding:12px;'
               'border-radius:6px;font-weight:bold">'
               '&#9888; No geometry detected. Survey needs a WKT geometry '
               'column or Latitude/Longitude columns.</div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()

_geom_type = gdf.geometry.geom_type.iloc[0]
print(f'{len(gdf)} features  |  geometry type: {_geom_type}')


## 4. Explore the Dataset

In [ ]:
# Select the variable to explore and map
numeric_cols = [c for c in gdf.columns
                if c != 'geometry' and pd.api.types.is_numeric_dtype(gdf[c])]

col_selector = widgets.Dropdown(
    options=numeric_cols,
    description='Variable:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='460px'),
)
display(HTML('<b>Select the variable to explore:</b>'))
display(col_selector)


In [ ]:
col_to_map = col_selector.value
print(f'Selected: {col_to_map}')

print('\n--- Dataset Info ---')
gdf.info()

print('\n--- Missing values (sorted) ---')
display(df.isnull().sum().sort_values(ascending=False)
          .to_frame('missing').query('missing > 0'))


In [ ]:
# Interactive choropleth map of the selected variable
cntry_name = gdf.columns[0]
gdf.explore(
    column=col_to_map, cmap='Blues', scheme='FisherJenks',
    tiles='CartoDB positron',
    tooltip=[cntry_name, col_to_map], popup=True, k=5, highlight=True,
    width='100%',
    legend_kwds={'caption': col_to_map, 'colorbar': False}
)


In [ ]:
# Histogram of the selected variable
fig, ax = plt.subplots(figsize=(8, 3))
gdf[col_to_map].dropna().hist(bins=30, ax=ax, edgecolor='black')
ax.set_title(col_to_map, fontsize=9)
ax.set_xlabel('')
plt.tight_layout()
plt.show()


## 5. Spatial Weights

### 4.1 Contiguity-based weights

Queen contiguity weights reflect adjacency as a binary indicator: $w_{ij}=1$ when polygons $i$ and $j$ share an edge or a vertex, 0 otherwise. For point data, 5-nearest-neighbor weights are used instead.

### 4.2 Distance-based weights

KNN weights use centroid distances to define neighbors. Run the cell below to choose the weight type and number of neighbors.

In [ ]:
# Create spatial weights
# For polygon data the default is Queen; for point data it is KNN(k=5).
# Adjust k below if needed.
_k = 5

if _geom_type == 'Point':
    wq = KNN.from_dataframe(gdf, k=_k)
    print(f'KNN weights (k={_k}): {wq.n} observations')
else:
    wq = Queen.from_dataframe(gdf)
    print(f'Queen weights: {wq.n} observations, '
          f'avg {wq.mean_neighbors:.1f} neighbors')

# Plot the weight network (matplotlib)
fig, ax = plt.subplots(figsize=(15, 5))
plot_spatial_weights(wq, gdf, ax=ax)
ax.set_title('Spatial weight network', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Explore neighbors of a selected feature
_name_col = gdf.columns[0]
neighbor_selector = widgets.BoundedIntText(
    value=0, min=0, max=len(gdf)-1, step=1,
    description='Feature index:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='280px'),
)
display(HTML(f'<b>Enter the integer index (0–{len(gdf)-1}) '
            f'of a feature to see its neighbors:</b>'))
display(neighbor_selector)


In [ ]:
# Show neighbors of the selected feature
_idx = neighbor_selector.value
_neighbors = wq.neighbors[_idx]
print(f'Feature {_idx} ({gdf.iloc[_idx][gdf.columns[0]]}) '
      f'has {len(_neighbors)} neighbor(s):')
for _n in _neighbors:
    print(f'  [{_n}]  {gdf.iloc[_n][gdf.columns[0]]}')


## 6. Spatial Lag

The **spatial lag** of observation $i$ is the weighted average of the attribute values of its neighbors: $y_{lag,i} = \sum_j w_{ij}\, y_j$. It measures how similar each unit is to its surroundings.

In [ ]:
# Drop rows without the selected variable, recompute row-standardised weights
gdfn = gdf.dropna(subset=[col_to_map]).copy()
y = gdfn[col_to_map]

if _geom_type == 'Point':
    _wlag = KNN.from_dataframe(gdfn, k=_k)
else:
    _wlag = Queen.from_dataframe(gdfn)
_wlag.transform = 'r'

ylag = libpysal.weights.lag_spatial(_wlag, y)
print(f'Spatial lag computed for {len(gdfn)} observations.')


In [ ]:
# Interactive map of the spatial lag variable
gdfn['_spatial_lag'] = ylag.values
gdfn.explore(
    column='_spatial_lag', cmap='Blues', scheme='FisherJenks',
    tiles='CartoDB positron',
    tooltip=[cntry_name, col_to_map, '_spatial_lag'],
    popup=True, k=5, highlight=True, width='100%',
    legend_kwds={'caption': f'Spatial lag of {col_to_map}', 'colorbar': False}
)


## 7. Join Counts (Binary Spatial Autocorrelation)

Join counts test for spatial autocorrelation in a binarised attribute. Each observation is labelled **Black** (value above median) or **White** (value at or below median). The test asks whether the observed number of Black–Black (BB) joins differs from what chance would produce.

- **BB** joins: both neighbours are Black
- **WW** joins: both neighbours are White
- **BW** joins: one Black, one White

In [ ]:
# Binarise: 1 = above median (Black), 0 = at or below (White)
_med = y.median()
yb = (y > _med).astype(int)
print(f'Median of {col_to_map}: {_med:.4f}')
print(f'Black (above median): {yb.sum()}  |  '
      f'White (at/below median): {(yb==0).sum()}')

# Map the binary variable
gdfn['_binary'] = yb.map({0: '0 Low', 1: '1 High'})
fig, ax = plt.subplots(figsize=(15, 5))
gdfn.plot(column='_binary', cmap='binary', edgecolor='grey',
          legend=True, ax=ax)
ax.set_title(f'{col_to_map}  —  High / Low split at median', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Join counts test (binary weights)
if _geom_type == 'Point':
    _wjc = KNN.from_dataframe(gdfn, k=_k)
else:
    _wjc = Queen.from_dataframe(gdfn)
_wjc.transform = 'b'
np.random.seed(12345)
jc = esda.join_counts.Join_Counts(yb, _wjc)

print(f'Observed BB (Black-Black) joins : {jc.bb:.0f}')
print(f'Observed WW (White-White) joins : {jc.ww:.0f}')
print(f'Observed BW (mixed) joins       : {jc.bw:.0f}')
print(f'Expected BB under CSR           : {jc.mean_bb:.2f}')
print(f'p-value (BB)                    : {jc.p_sim_bb:.4f}')

if jc.p_sim_bb < 0.05:
    print('\u2192 Significant clustering of High values (p < 0.05)')
else:
    print('\u2192 No significant clustering detected (p >= 0.05)')


## 8. Moran's I (Global Spatial Autocorrelation)

Moran's I measures global spatial autocorrelation for a continuous variable. The Moran scatter plot shows the standardised values against their spatial lag; the slope of the regression line is Moran's I. The reference distribution is built from 999 random spatial permutations.

In [ ]:
# Row-standardised weights for Moran's I
if _geom_type == 'Point':
    _wmi = KNN.from_dataframe(gdfn, k=_k)
else:
    _wmi = Queen.from_dataframe(gdfn)
_wmi.transform = 'r'
np.random.seed(12345)

mi = Moran(y, _wmi)
print(f"Moran's I : {mi.I:.4f}")
print(f'p-value   : {mi.p_sim:.4f}')
if mi.p_sim < 0.05:
    direction = 'positive' if mi.I > 0 else 'negative'
    print(f'\u2192 Significant {direction} spatial autocorrelation (p < 0.05)')
else:
    print('\u2192 No significant spatial autocorrelation (p >= 0.05)')

plot_moran(mi, zstandard=True, figsize=(15, 5))
plt.suptitle(f"Moran's I for {col_to_map}", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()


## 9. Local Spatial Autocorrelation (LISA)

Local Indicators of Spatial Association (LISA) decompose the global Moran's I into a per-observation contribution. Each observation is classified into one of four quadrants:

- **HH** (Hot spot): high value surrounded by high neighbours
- **LL** (Cold spot): low value surrounded by low neighbours
- **HL** (Spatial outlier): high value surrounded by low neighbours
- **LH** (Spatial outlier): low value surrounded by high neighbours

Only observations with p < 0.05 (999 permutations) are coloured; all others are shown in light grey.

In [ ]:
np.random.seed(12345)
lisa = Moran_Local(y, _wmi)

# Quadrant labels (only show significant at p < 0.05)
_sig = lisa.p_sim < 0.05
_quad_labels = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
_colors      = {'HH': '#d7191c', 'LL': '#2c7bb6',
                'HL': '#fdae61', 'LH': '#abd9e9', 'ns': '#d3d3d3'}

gdfn['_lisa_q']     = lisa.q
gdfn['_lisa_p']     = lisa.p_sim
gdfn['_lisa_label'] = [
    _quad_labels[q] if sig else 'ns'
    for q, sig in zip(lisa.q, _sig)
]

# Summary
for _lbl in ['HH', 'LL', 'HL', 'LH', 'ns']:
    _n = (gdfn['_lisa_label'] == _lbl).sum()
    print(f'  {_lbl:3s}: {_n}')

# Matplotlib map
fig, ax = plt.subplots(figsize=(15, 6))
for _lbl, _col in _colors.items():
    _mask = gdfn['_lisa_label'] == _lbl
    gdfn[_mask].plot(color=_col, edgecolor='0.5', linewidth=0.3,
                    ax=ax, label=_lbl)
ax.legend(title='LISA cluster', loc='lower right', fontsize=8)
ax.set_title(f'LISA cluster map  —  {col_to_map}', fontsize=10)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## 10. OLS Regression with Spatial Diagnostics

The `spreg` OLS implementation adds spatial diagnostics to a standard linear regression: Moran's I on residuals, Lagrange Multiplier tests (LM-lag, LM-error), and robust variants. Independent variables are standardised (zero mean, unit variance) before fitting.

Select the dependent variable $y$ and one or more independent variables $X$, then run the computation cell.

In [ ]:
# Select dependent and independent variables
dep_sel = widgets.Dropdown(
    options=numeric_cols,
    description='Dependent (y):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='480px'),
)
ind_sel = widgets.SelectMultiple(
    options=numeric_cols,
    value=numeric_cols[:min(3, len(numeric_cols))],
    description='Independent (X):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='480px', height='180px'),
    rows=8,
)
display(HTML('<b>Select dependent and independent variables, '
            'then run the next cell.</b><br>'
            'Hold Ctrl/Cmd to select multiple independent variables.'))
display(dep_sel)
display(ind_sel)


In [ ]:
from spreg.ols import OLS
from sklearn.preprocessing import StandardScaler

dep_var = dep_sel.value
ind_vars = [v for v in list(ind_sel.value) if v != dep_var]
if not ind_vars:
    raise ValueError('Select at least one independent variable.')
print(f'y = {dep_var}')
print(f'X = {ind_vars}')

# Drop rows with missing values in any selected column
gdfn2 = gdfn.dropna(subset=[dep_var] + ind_vars).copy()
print(f'{len(gdfn2)} rows after dropping missing values '
      f'(dropped {len(gdfn) - len(gdfn2)})')

# Standardise X
x2  = gdfn2[ind_vars]
x2s = StandardScaler().fit_transform(x2.values)
y2  = gdfn2[dep_var]

# Recompute row-standardised weights for the regression subset
if _geom_type == 'Point':
    _wols = KNN.from_dataframe(gdfn2, k=_k)
else:
    _wols = Queen.from_dataframe(gdfn2)
_wols.transform = 'R'

# OLS with spatial diagnostics
m1 = OLS(
    y2.values[:, None], x2s,
    w=_wols,
    spat_diag=True, moran=True,
    name_x=ind_vars, name_y=dep_var
)
print(m1.summary)


## 11. Map OLS Residuals

Mapping residuals helps identify spatial patterns that the model missed.

In [ ]:
gdfn2['_ols_resid'] = m1.u.flatten()
gdfn2.explore(
    column='_ols_resid', cmap='RdYlBu', scheme='FisherJenks',
    tiles='CartoDB positron',
    tooltip=[cntry_name, dep_var, '_ols_resid'],
    popup=True, k=5, highlight=True, width='100%',
    legend_kwds={'caption': 'OLS Residuals', 'colorbar': False}
)
